# Columns added to the final fibroblast/decidual objects (Ç. Cevrim, 13 Nov 2026)

Reads the two objects and writes **new** files with these obs columns added; the input files are only read, and
the notebook stops if an output path exists or is one of the inputs.

| object | column | from |
|---|---|---|
| human | `pseudotime` | `20261107_Human_fibdec_final_nick_pdt.h5ad` (the pseudotime of record) |
| human | `transferred_mouse_fibdec`, `transferred_mouse_pseudotime` | `20261111_samap_final.h5ad` (SAMap of record) |
| mouse | `pseudotime` | `20261110_mouse_only_eFib_dec_final_nick_pdt.h5ad` (the pseudotime of record) |
| mouse | `Ulrich2024_sg_Stro7_top50_transferred` | the CSV of `stro7_score_transfer.ipynb` |

Values are copied as stored, matched by cell name. Every other part of the objects is written back unchanged.


In [1]:
# Parameters (papermill -p NAME VALUE overrides them)
D = "/n/lab_storage/eddy_lab/Lab/mckinley/data"
HUMAN_IN = f"{D}/20261111_Human_only_eFib_dec_final.h5ad"
MOUSE_IN = f"{D}/20261111_mouse_only_eFib_dec_final.h5ad"
HUMAN_PSEUDOTIME = f"{D}/20261112_Human_fibdec_final_nick_pdt.h5ad"
MOUSE_PSEUDOTIME = f"{D}/20261110_mouse_only_eFib_dec_final_nick_pdt.h5ad"
SAMAP = f"{D}/20261113_samap_final_ensembl_ids.h5ad"
STRO7_CSV = "./stro7_top50_20261113/Ulrich2024_sg_Stro7_top50_transferred_mouse.csv"  # Ulrich2024_sg_Stro7_top50_transferred_mouse.csv from stro7_score_transfer.ipynb
HUMAN_OUT = f"{D}/20261113_Human_only_eFib_dec_final_nick.h5ad"  # a new file, e.g. f"{D}/20261111_Human_only_eFib_dec_final_nick.h5ad"
MOUSE_OUT = f"{D}/20261113_mouse_only_eFib_dec_final_nick.h5ad"  # a new file, e.g. f"{D}/20261111_mouse_only_eFib_dec_final_nick.h5ad"


In [2]:
from pathlib import Path

import anndata as ad
import h5py
import pandas as pd
from anndata.io import read_elem

inputs = [HUMAN_IN, MOUSE_IN, HUMAN_PSEUDOTIME, MOUSE_PSEUDOTIME, SAMAP, STRO7_CSV]
for out in (HUMAN_OUT, MOUSE_OUT):
    assert out and not Path(out).exists(), f"set a new output file (got {out!r})"
    assert Path(out).resolve() not in {Path(path).resolve() for path in inputs}, f"{out} is an input"


def obs_columns(path, columns):
    with h5py.File(path, "r") as handle:
        obs = read_elem(handle["obs"])
    obs.index = obs.index.astype(str)
    return obs[columns]


samap = obs_columns(SAMAP, ["species", "transferred_mouse_fibdec", "transferred_mouse_pseudotime"])
added = {
    "human": pd.concat([obs_columns(HUMAN_PSEUDOTIME, ["pseudotime"]),
                        samap.loc[samap["species"] == "hs", ["transferred_mouse_fibdec",
                                                             "transferred_mouse_pseudotime"]]], axis=1),
    "mouse": pd.concat([obs_columns(MOUSE_PSEUDOTIME, ["pseudotime"]),
                        pd.read_csv(STRO7_CSV, index_col="cell")], axis=1),
}
for species, columns in added.items():
    print(f"{species}: {len(columns):,} cells; values per column:", columns.notna().sum().to_dict())


AssertionError: set a new output file (got '/n/lab_storage/eddy_lab/Lab/mckinley/data/20261113_Human_only_eFib_dec_final_nick.h5ad')

## The new objects

In [ ]:
for species, source, out in (("human", HUMAN_IN, HUMAN_OUT), ("mouse", MOUSE_IN, MOUSE_OUT)):
    adata = ad.read_h5ad(source)
    columns = added[species]
    assert set(columns.index) == set(adata.obs_names) and len(columns) == adata.n_obs, f"{species}: cells differ"
    assert not set(columns.columns) & set(adata.obs.columns), f"{species}: a column is already in {source}"
    for column in columns.columns:
        adata.obs[column] = columns[column].reindex(adata.obs_names)
    adata.write_h5ad(out, compression="gzip", convert_strings_to_categoricals=False)
    print(f"wrote {out}: {adata.n_obs:,} cells; added {list(columns.columns)}")


## Check by eye

Read back from the new files: each added column's number of values. The rest of each file can be compared with its
input array by array (`tmp/2026_11_11_prove_identical/compare_h5ad_arrays.py IN OUT`: only the added columns differ).

In [ ]:
for species, out in (("human", HUMAN_OUT), ("mouse", MOUSE_OUT)):
    written = obs_columns(out, list(added[species].columns))
    print(species, written.notna().sum().to_dict())
